# Agent Architecture Comparison

Compare SFT, RL, BC+RL, and Agentic LLM agents across multiple dimensions.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# Connect to database
db_path = Path('data/game_data.db')
conn = sqlite3.connect(db_path)

# Load agent results
results_df = pd.read_sql_query(
    'SELECT * FROM agent_results ORDER BY agent_name',
    conn
)

print(f'Loaded {len(results_df)} agent results')
print(f'\nAgents: {results_df["agent_name"].unique()}')
print(f'\nFirst few rows:\n{results_df.head()}')

## Win Rate Comparison

In [ ]:
# Summary statistics by agent
print('='*70)
print('AGENT PERFORMANCE SUMMARY')
print('='*70)

for agent in results_df['agent_name'].unique():
    agent_data = results_df[results_df['agent_name'] == agent]
    print(f'\n{agent.upper()}')
    print(f'  Games Played:      {agent_data["games_played"].sum():,}')
    print(f'  Wins:              {agent_data["wins"].sum():,}')
    print(f'  Losses:            {agent_data["losses"].sum():,}')
    print(f'  Ties:              {agent_data["ties"].sum():,}')
    print(f'  Win Rate:          {agent_data["win_rate"].mean():.2%}')
    print(f'  Fidelity:          {agent_data["behavioral_fidelity"].mean():.2%}')
    print(f'  Action KL:         {agent_data["action_kl"].mean():.3f}')
    print(f'  Avg Decision (ms): {agent_data["avg_decision_ms"].mean():.1f}')

In [ ]:
# Win rate comparison visualization
fig = px.bar(
    results_df.groupby('agent_name')['win_rate'].mean().reset_index(),
    x='agent_name',
    y='win_rate',
    title='Average Win Rate by Agent',
    labels={'agent_name': 'Agent', 'win_rate': 'Win Rate'},
    color='win_rate',
    color_continuous_scale='Viridis'
)
fig.show()

## Behavioral Fidelity Analysis

In [ ]:
# Fidelity comparison (how well agents mimic human play)
fig = px.box(
    results_df,
    x='agent_name',
    y='behavioral_fidelity',
    title='Behavioral Fidelity Distribution',
    labels={'agent_name': 'Agent', 'behavioral_fidelity': 'Fidelity (%)'},
    color='agent_name'
)
fig.show()

print('\nBehavioral Fidelity Analysis:')
print(results_df.groupby('agent_name')['behavioral_fidelity'].agg(['mean', 'std', 'min', 'max']))

## Action Distribution KL Divergence

## Fidelity vs Performance Tradeoff

In [ ]:
# Key insight: Are high-fidelity agents worse at winning?
fig = px.scatter(
    results_df,
    x='behavioral_fidelity',
    y='win_rate',
    color='agent_name',
    size='avg_decision_ms',
    hover_data=['games_played', 'action_kl'],
    title='Fidelity vs Performance Tradeoff (size = latency)',
    labels={
        'behavioral_fidelity': 'Behavioral Fidelity',
        'win_rate': 'Win Rate',
        'avg_decision_ms': 'Decision Latency (ms)'
    }
)
fig.show()

# Correlation analysis
print('\nCorrelation: Fidelity vs Win Rate')
corr = results_df['behavioral_fidelity'].corr(results_df['win_rate'])
print(f'Correlation coefficient: {corr:.3f}')
if corr < -0.3:
    print('⚠️ Strong negative correlation: High fidelity agents have lower win rates')
elif corr > 0.3:
    print('✅ Positive correlation: Fidelity and performance aligned')
else:
    print('📊 Weak correlation: Fidelity and performance are independent')

## Latency Comparison

In [ ]:
# Decision latency by agent
fig = px.bar(
    results_df.groupby('agent_name')['avg_decision_ms'].mean().reset_index(),
    x='agent_name',
    y='avg_decision_ms',
    title='Average Decision Latency',
    labels={'agent_name': 'Agent', 'avg_decision_ms': 'Latency (ms)'},
    color='avg_decision_ms',
    color_continuous_scale='Reds'
)
fig.show()

print('\nLatency Analysis:')
print(results_df.groupby('agent_name')['avg_decision_ms'].agg(['mean', 'std', 'min', 'max']))

## Multi-Dimensional Comparison

In [ ]:
# Radar chart comparison
agents = results_df['agent_name'].unique()

# Normalize metrics to 0-100 scale
metrics_data = []
for agent in agents:
    agent_data = results_df[results_df['agent_name'] == agent].iloc[0]
    
    # Normalize
    win_rate_norm = agent_data['win_rate'] * 100
    fidelity_norm = agent_data['behavioral_fidelity'] * 100
    kl_norm = 100 - (agent_data['action_kl'] / results_df['action_kl'].max() * 100)  # invert
    latency_norm = 100 - (agent_data['avg_decision_ms'] / results_df['avg_decision_ms'].max() * 100)  # invert
    
    metrics_data.append({
        'Agent': agent,
        'Win Rate': win_rate_norm,
        'Fidelity': fidelity_norm,
        'Action Alignment': kl_norm,
        'Speed': latency_norm
    })

metrics_df = pd.DataFrame(metrics_data)

# Radar chart
fig = go.Figure()

for _, row in metrics_df.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=[row['Win Rate'], row['Fidelity'], row['Action Alignment'], row['Speed']],
        theta=['Win Rate', 'Fidelity', 'Action Alignment', 'Speed'],
        fill='toself',
        name=row['Agent']
    ))

fig.update_layout(title='Multi-Dimensional Agent Comparison')
fig.show()

## Statistical Significance Testing

## Key Findings & Recommendations